In [ ]:
from datetime import datetime
from pathlib import Path
import glob
import os

import geopandas as gpd
from IPython.display import display, HTML
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_poisson_deviance, r2_score
import xgboost as xgb

from thomas2024.calibration import assemble_thresholds, lookup_pop_at_risk, lookup_pop_exposed


display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

In [ ]:
storm_set_params = {
    "STORM_constant": ("STORM", 2000, None, None),
    "STORM_CNRM-CM6-1-HR": ("STORM", 2050, "RCP8.5", "CNRM"),
    "STORM_CMCC-CM2-VHR4": ("STORM", 2050, "RCP8.5", "CMCC"),
    "STORM_EC-Earth3P-HR": ("STORM", 2050, "RCP8.5", "EC-Earth"),
    "STORM_HadGEM3-GC31-HM": ("STORM", 2050, "RCP8.5", "HadGEM3"),

    "IRIS_PRESENT": ("IRIS", 2000, None, None),
    "IRIS_SSP5-2050": ("IRIS", 2050, "SSP5-8.5", None),

    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2010": ("CHAZ", 2010, None, "UKESM1"),
    "CHAZ_SSP-585_GCM-CESM2_epoch-2010": ("CHAZ", 2010, None, "CESM2"),
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2010": ("CHAZ", 2010, None, "CNRM"),
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2010": ("CHAZ", 2010, None, "EC-Earth3"),
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2010": ("CHAZ", 2010, None, "MIROC6"),
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2050": ("CHAZ", 2050, "SSP5-8.5", "UKESM1"),
    "CHAZ_SSP-585_GCM-CESM2_epoch-2050": ("CHAZ", 2050, "SSP5-8.5", "CESM2"),
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2050": ("CHAZ", 2050, "SSP5-8.5", "CNRM"),
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2050": ("CHAZ", 2050, "SSP5-8.5", "EC-Earth3"),
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2050": ("CHAZ", 2050, "SSP5-8.5", "MIROC6"),

    "emanuel_ssp-585_gcm-cesm2_epoch-2005": ("Emanuel", 2005, None, "CESM2"),
    "emanuel_ssp-585_gcm-cnrm6_epoch-2005": ("Emanuel", 2005, None, "CNRM6"),
    "emanuel_ssp-585_gcm-ecearth6_epoch-2005": ("Emanuel", 2005, None, "EC-Earth6"),
    "emanuel_ssp-585_gcm-miroc6_epoch-2005": ("Emanuel", 2005, None, "MIROC6"),
    "emanuel_ssp-585_gcm-ukmo6_epoch-2005": ("Emanuel", 2005, None, "UKMO6"),
    "emanuel_ssp-585_gcm-cesm2_epoch-2050": ("Emanuel", 2050, "SSP5-8.5", "CESM2"),
    "emanuel_ssp-585_gcm-cnrm6_epoch-2050": ("Emanuel", 2050, "SSP5-8.5", "CNRM6"),
    "emanuel_ssp-585_gcm-ecearth6_epoch-2050": ("Emanuel", 2050, "SSP5-8.5", "EC-Earth6"),
    "emanuel_ssp-585_gcm-miroc6_epoch-2050": ("Emanuel", 2050, "SSP5-8.5", "MIROC6"),
    "emanuel_ssp-585_gcm-ukmo6_epoch-2050": ("Emanuel", 2050, "SSP5-8.5", "UKMO6"),
}

group_colours = {
    "IRIS": "lightgreen",
    "STORM": "plum",
    "CHAZ": "pink",
    "MIT": "lightblue",
}

In [ ]:
# Input data paths
country_thresholds_path = Path("data/20260907_country_thresholds.csv")
income_thresholds_path = Path("data/20260907_income_thresholds.csv")
wb_income_path = Path("data/OGHIST_2026_07_15.xlsx")
results_dir = Path("/data/tc-grid/open-gira/results")

# Output paths
plot_dir = Path("../plots")

thresholds_by_country = assemble_thresholds(wb_income_path, income_thresholds_path, country_thresholds_path)

metrics = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(str(results_dir / Path("power/by_country/*/network/metrics.pq"))))])
metrics = metrics.dropna(how="all")
metrics["gamma_index"] = metrics["gamma_index"].astype(float)
metrics = metrics.rename(columns={c: f"net_{c}" for c in metrics.columns})

nodes = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(str(results_dir / Path("power/by_country/*/network/nodes.geoparquet"))))])
node_pop = nodes.loc[:, ["iso_a3", "population"]].dropna().groupby("iso_a3").sum().rename(columns={"population": "total_pop"})

risk = []
for atmosphere, (model, epoch, scenario, gcm) in storm_set_params.items():
    df = pd.read_parquet(results_dir / f"power/by_storm_set/{atmosphere}/disruption/drivers.pq")
    exposure = pd.read_parquet(results_dir / f"power/by_storm_set/{atmosphere}/exposure/people_exposed_by_event.pq")
    exposure = exposure.rename(columns={c: f"pop_{c}" for c in exposure.columns if c not in ["event_id", "iso_a3"]})
    exposure = lookup_pop_exposed(exposure, thresholds_by_country)
    df = df.merge(exposure, on=["event_id", "iso_a3"], how="left")
    df["model"] = model
    df["epoch"] = epoch
    df["scenario"] = scenario
    df["gcm"] = gcm
    risk.append(lookup_pop_at_risk(df, thresholds_by_country))

df = pd.concat(risk)
df = df.merge(metrics.reset_index(), on="iso_a3")
df = df.merge(node_pop.reset_index(), on="iso_a3")
df["wind_exceedance_ms-1"] = df["max_wind_speed_ms"] - df["threshold_ms-1"]
df["pop_at_risk_ratio"] = df["pop_at_risk"] / df["total_pop"]

# Filter out non-events!
df = df[df["pop_at_risk_ratio"] > 0]

In [ ]:
t0 = datetime.now()

# Feature selection
TARGET = "pop_at_risk_ratio"
GROUP_COL = "event_id"
drop_cols = [GROUP_COL, "pop_at_risk", "pop_at_risk_ratio", "total_pop"]
cat_cols = ["iso_a3", "storm_set", "model", "epoch", "scenario", "gcm", "income_level"]
features = [c for c in df.columns if c not in drop_cols]
mask = df[TARGET].notna() & np.isfinite(df[TARGET])
print(f"Dropping {(~mask).sum()} of {len(df)} rows")
df = df[mask].reset_index(drop=True)
X = df[features].copy()
y = df[TARGET]
groups = df[GROUP_COL]
for c in cat_cols:
    if c in X.columns:
        X[c] = X[c].astype("category")

# Do cross-validation with group k fold
gkf = GroupKFold(n_splits=5)
oof_pred = np.zeros(len(df))
params = dict(
    objective="reg:tweedie",
    tweedie_variance_power=1.3,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    tree_method="hist",
)
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
    model = xgb.XGBRegressor(**params, early_stopping_rounds=30)
    model.fit(
        X.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=[(X.iloc[va_idx], y.iloc[va_idx])],
        verbose=False,
    )
    oof_pred[va_idx] = model.predict(X.iloc[va_idx])
print("OOF R2:", r2_score(y, oof_pred))
print("OOF Poisson deviance:", mean_poisson_deviance(y.clip(lower=1e-6), oof_pred.clip(min=1e-6)))

# Fit final model using all data
final_model = xgb.XGBRegressor(**{k: v for k, v in params.items()})
final_model.fit(X, y)
explainer = shap.TreeExplainer(
    model
)
shap_values = explainer(X)
print("In-sample R2:", r2_score(y, final_model.predict(X)))
baseline_pred = df.groupby("iso_a3")[TARGET].transform("mean")
print("Country-mean baseline R2:", r2_score(y, baseline_pred))
print(f"Took: {datetime.now() - t0}")

In [ ]:
shap.plots.beeswarm(shap_values, max_display=20)

In [ ]:
shap.plots.bar(shap_values, max_display=20)

In [ ]:
explanation = explainer(X[::100])
shap.plots.scatter(explanation[:, ["wind_exceedance_ms-1", "max_wind_speed_ms", "min_pressure_hpa", "model", "net_total_length_km", "net_n_communities", "net_betweenness_gini"]])

In [ ]:
percentiles = [90, 95, 99, 99.5, 99.9]

# find the row closest to each percentile value of y
target_vals = np.percentile(y, percentiles)
idxs = [np.abs(y - t).idxmin() for t in target_vals]

for p, idx in zip(percentiles, idxs):
    pos = X.index.get_loc(idx)
    print(f"--- {p}th percentile | actual={y.loc[idx]:.4f} | pred={final_model.predict(X.loc[[idx]])[0]:.4f} ---")
    shap.plots.force(
        explainer.expected_value,
        shap_values.values[pos],
        X.iloc[pos],
        matplotlib=True,
    )